# V2-2 và Qwen3-VL-4B: cùng 47 ca test, cùng user prompt

Chạy inference Qwen3-VL chưa fine-tune, chỉ nhận text grading. Đối chiếu với dự đoán V2-2 đã lưu, được khóa bằng SHA-256; cả hai được chấm lại cùng metrics. Đây là so sánh hai hệ thống khác model nền, chưa cô lập tác động fine-tune.

Notebook phải private, bật GPU T4 và Internet. Thêm Input → Notebook Output của `kieuthithutrang/mri-v2-2-evaluation`. Chưa có kết quả Qwen3 cho tới khi chạy xong các cell.

## 1. Tải code và chuẩn bị môi trường

Giữ nguyên bản CUDA PyTorch của Kaggle. Cài phiên bản Transformers hỗ trợ Qwen3-VL trong runtime riêng của notebook này.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
CODE_SHA = "e5956211e94c9c5feea92af80cf87c164c90bf62"
REPO = Path("/kaggle/working/repo")
RUN_DIR = Path("/kaggle/working/runs/v2-2-qwen3-comparison-01")
os.environ["HF_HOME"] = "/tmp/mri-qwen3-hf"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
assert not REPO.exists(), "Dùng phiên Kaggle mới để giữ môi trường tái lập"
setup_start = time.monotonic()
subprocess.run(["git", "clone", "https://github.com/kttt294/MRI-report-generator.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", CODE_SHA], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-qwen3-evaluation.txt")], check=True)
print("Code:", CODE_SHA, "Setup seconds:", round(time.monotonic() - setup_start, 1))

## 2. Kiểm tra dữ liệu, sinh 47 ca và chấm điểm

Một lượt mỗi ca; greedy; 1.536 token; repetition penalty 1.05 kế thừa lượt V2-2 cũ. Không retry hoặc fallback. Log cập nhật thời gian còn lại sau từng ca. Script dừng nếu dữ liệu hoặc kết quả cũ không đúng hash.

In [ ]:
subprocess.run([sys.executable, "-u", "scripts/compare_v2_2_qwen3.py",
                "--input-root", "/kaggle/input",
                "--config", "configs/v2_2_qwen3_comparison.json",
                "--output", str(RUN_DIR)], cwd=REPO, check=True)

## 3. Bảng so sánh

BLEU-4 dùng thang 0–100. Các điểm F1 giữ giá trị gốc. Ca JSON lỗi vẫn nằm trong mẫu số; chấm đầu ra rỗng với BLEU/ROUGE và 0 với BERTScore. Thời gian của model fine-tune lấy từ lần chạy cũ, nên chưa phải đo tốc độ cùng phiên/phần mềm.

In [ ]:
import pandas as pd
from IPython.display import display
comparison = pd.read_csv(RUN_DIR / "comparison.csv")
display(comparison)
print("Kết quả:", RUN_DIR)
print("Bảng bác sĩ đọc đối chiếu, có cả output thô:", RUN_DIR / "human_review.csv")
print("JSON hợp lệ và NLP metrics không xác nhận đúng y khoa. Fold 1 test đã được xem trước đây; kết quả mang tính thăm dò.")